# U-Net ile kan hucresi segmentasyonu (Colab)

**Once:** Menuden `Runtime > Change runtime type > T4 GPU` secin.

Drive'da beklenen yapi:
```
MyDrive/
  SEG/                  <- paylasilan klasore "Drive'ima kisayol ekle" ile
    train/7.png ...
    maske/7.nii ...
  unet_segmentation/    <- bilgisayardaki kod klasoru (train.py'nin oldugu klasor)
```
Yollariniz farkliysa sadece asagidaki ayar hucresini degistirin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1) Ayarlar

In [ ]:
CODE_DIR  = '/content/drive/MyDrive/unet_segmentation'   # train.py'nin oldugu klasor
IMAGE_DIR = '/content/drive/MyDrive/SEG/train'           # PNG goruntuler
MASK_DIR  = '/content/drive/MyDrive/SEG/maske'           # NIfTI maskeler
DATA_DIR  = '/content/veri'                              # hazirlanan veri (Colab yerel diski, hizli)
CKPT_DIR  = CODE_DIR + '/checkpoints_kan'                # model Drive'a kaydedilir, oturum kapansa da kalir

import os
for p in [CODE_DIR, IMAGE_DIR, MASK_DIR]:
    print('OK ' if os.path.isdir(p) else 'YOK', p)

## 2) Kurulum

In [ ]:
%cd {CODE_DIR}
!pip install -q -r requirements.txt nibabel
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3) Veriyi hazirla
PNG + `.nii` maskeleri `images/` + `masks/` PNG ciftlerine cevirir ve 4K goruntuleri 960x540'a kucultur.
Drive'dan ~2.5 GB okundugu icin birkac dakika surebilir; oturum basina bir kez calistirmaniz yeterli.

In [ ]:
!python prepare_data.py --images "{IMAGE_DIR}" --masks "{MASK_DIR}" --out "{DATA_DIR}" --scale 0.25

In [ ]:
# Kontrol: bir goruntu ve maskesi ust uste
import cv2, matplotlib.pyplot as plt
img = cv2.imread(f'{DATA_DIR}/images/7.png')[:, :, ::-1]
msk = cv2.imread(f'{DATA_DIR}/masks/7.png', 0)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].imshow(img); ax[0].set_title('Goruntu')
ax[1].imshow(img); ax[1].imshow(msk, alpha=0.4, cmap='Greens'); ax[1].set_title('Maske')
for a in ax: a.axis('off')
plt.show()

## 4) Egitim
73 egitim / 18 dogrulama ornegi. T4 GPU'da 100 epoch yaklasik 10-15 dakika surer.
En iyi model `CKPT_DIR/best_unet.pt` olarak kaydedilir.

In [ ]:
!python train.py --data-root "{DATA_DIR}" --checkpoint-dir "{CKPT_DIR}"     --epochs 100 --batch-size 8 --img-size 512 --base-channels 32 --num-workers 2

## 5) Sonuc

In [ ]:
from IPython.display import Image, display
display(Image(f'{CKPT_DIR}/sample_prediction.png'))